In [ ]:
!pip install --upgrade transformers huggingface_hub

import os
os.makedirs('./model', exist_ok=True)
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [ ]:
%%writefile teacher_trainer.py
import argparse
import random
import torch
from torch.utils.data import DataLoader, Subset, Dataset
from torchvision import datasets, transforms
from torchmetrics.classification import Accuracy
from transformers import get_linear_schedule_with_warmup
from tqdm import tqdm
from sklearn.datasets import fetch_20newsgroups
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification
from datasets import load_dataset
from evaluation import mc_cal


class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]
        encoding = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_len,
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0).bool(),
            'label': torch.tensor(label, dtype=torch.long)
        }





def train_teacher(model, config, known_unknown=0, unknown_unknown=1):

    allowed_indiced = []
    for i in range(14):
        if i not in [known_unknown, unknown_unknown]:
            allowed_indiced.append(i)

    print(allowed_indiced)
    batch_size = 64
    num_epochs = 1
    T = config['T']
    M = config['M']
    R = config['R']
    p = config['p']
    rw= config['rw']

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')

    dataset = load_dataset("dbpedia_14")

    train_full = dataset['train']
    train_full = TextDataset(train_full['content'], train_full['label'], tokenizer, max_len=64)
    

    test_ds = dataset['test']
    test_ds = TextDataset(test_ds['content'], test_ds['label'], tokenizer, max_len=64)


    
    train_indices = [i for i, data in enumerate(train_full) if data['label'] in allowed_indiced]
    train_ds = Subset(train_full, train_indices)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
    test_loader = DataLoader(test_ds, batch_size=batch_size, num_workers=2, pin_memory=True)


    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=2e-5,
        weight_decay=0.01,
        betas=(0.9, 0.999),
        eps=1e-8
    )

    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=1, gamma=0.9)

    criterion = nn.CrossEntropyLoss()

    accuracy = Accuracy(task="multiclass", num_classes=12).to(device)


    for epoch in range(1, num_epochs + 1):
        loop = tqdm(train_loader, desc=f"Epoch {epoch}/{num_epochs}", leave=False)
        running_loss, running_acc = 0.0, 0.0
        for data in loop:
            x = data['input_ids'].to(device)
            y = data['label'].to(device)
            att = data['attention_mask'].to(device)
            index = torch.randint(0, M, [len(y)])

            for i in range(y.shape[0]):
                y[i] = allowed_indiced.index(y[i])

            optimizer.zero_grad()
            logits = model(x, att, index)

            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * x.size(0)
            running_acc += accuracy(logits, y) * x.size(0)
            loop.set_postfix(loss=f"{loss.item():.4f}")

        epoch_loss = running_loss / len(train_loader.dataset)
        epoch_acc = running_acc / len(train_loader.dataset)


        print(f"Epoch {epoch:02d}/{num_epochs} - train loss: {epoch_loss:.4f} - train acc: {epoch_acc:.2f}")
        scheduler.step()
        torch.save(model.state_dict(), "./model/teacher_model.pth")



    model.train()
    with torch.no_grad():
        result = mc_cal(model, test_loader, allowed_indiced, T=T, M=M, R=R, num_classes=14)
        with open(f"log-{known_unknown}{unknown_unknown}-{p}-{rw}.txt", "a") as f:
            print(result, file=f)
        print(result)

    torch.save(model.state_dict(), "./model/teacher_model.pth")



In [ ]:
%%writefile teacher.py
import torch
import torch.nn as nn
from transformers import DistilBertForSequenceClassification


class ControlledDropout(nn.Module):
    def __init__(self, p, shape, device, M=10, rw=.5, indices=None):
        super(ControlledDropout, self).__init__()
        self.store = []
        for i in range(M):
            mask = torch.empty(shape, device=device).bernoulli_(1 - p)
            self.store.append(mask)

        self.rw = rw
        self.p = p
        self.index = indices

    def forward(self, x):

        selected = torch.stack([self.store[i] for i in self.index[0]], dim=0)
        random_walk = torch.ones_like(selected, device=x.device) * self.rw
        random_walk = random_walk + selected * (2*self.p-1)/(1-self.p) * self.rw
        random_walk = torch.bernoulli(random_walk)

        mask = (selected.int() ^ random_walk.int()) / (1 - self.p)

        return x * mask

    def get_candidates(self, indices):
        return [self.store[i] for i in indices]





class TeacherModel(nn.Module):

    def __init__(self, config):

        super().__init__()

        self.base = DistilBertForSequenceClassification.from_pretrained(
            config['model_name'],
            num_labels=config['num_classes']
        )


        self.mask_indices = [None]
        shape = [64, 768]

        self.base.distilbert.embeddings.dropout = ControlledDropout(config['p'], shape, config['device'], M=config['M'], rw=config['rw'], indices=self.mask_indices)

        for i, layer in enumerate(self.base.distilbert.transformer.layer):
            layer.attention.dropout = ControlledDropout(config['p'], shape, config['device'], M=config['M'], rw=config['rw'], indices=self.mask_indices)
            layer.ffn.dropout = ControlledDropout(config['p'], shape, config['device'], M=config['M'], rw=config['rw'], indices=self.mask_indices)

        self.base.dropout = ControlledDropout(config['p'], [768], config['device'], M=config['M'], rw=config['rw'], indices=self.mask_indices)


    def forward(self, x, att, mask_indices):

        self.mask_indices.pop()
        self.mask_indices.append(mask_indices)

        output = self.base(x, attention_mask=att)

        return output.logits


    def get_candidates(self, indices):
        masks = {}

        masks['dropout_emb'] = torch.stack(self.base.distilbert.embeddings.dropout.get_candidates(indices))

        masks['dropout_att'] = {}
        masks['dropout_ffn'] = {}
        for i, layer in enumerate(self.base.distilbert.transformer.layer):
            masks['dropout_att'][i] = torch.stack(layer.attention.dropout.get_candidates(indices))
            masks['dropout_ffn'][i] = torch.stack(layer.ffn.dropout.get_candidates(indices))

        masks['dropout_cls'] = torch.stack(self.base.dropout.get_candidates(indices))
        masks['dropout_cls'] = masks['dropout_cls'].unsqueeze(dim=1)

        return masks




In [ ]:
%%writefile student.py
import torch
import torch.nn as nn
import numpy as np
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification



class SoftDropout(nn.Module):
    def __init__(self, p, ref, LCMC=True):
        super(SoftDropout, self).__init__()

        self.agg = nn.Conv2d(3, 1, kernel_size=1, bias=True)
        self.masks = ref
        self.p = p
        self.LCMC = LCMC

    def forward(self, x):

        if not self.LCMC:
            return x

        mask = self.agg(self.masks[0])
        mask = F.sigmoid(mask) / (1-self.p)

        if mask.shape[0]==1 and mask.shape[1]==1:
            mask = mask.squeeze(dim=0)

        return x * mask





class StudentModel(nn.Module):

    def __init__(self, config):

        super().__init__()

        self.base = DistilBertForSequenceClassification.from_pretrained(
            config['model_name'],
            num_labels=config['num_classes']
        )


        self.mask_storage = {
            "dropout_emb":[], 
            "dropout_att": {i:[] for i in range(len(self.base.distilbert.transformer.layer))},
            "dropout_ffn": {i:[] for i in range(len(self.base.distilbert.transformer.layer))},
            "dropout_cls": []
        }

        p = config['p']


        self.base.distilbert.embeddings.dropout = SoftDropout(p, ref=self.mask_storage['dropout_emb'], LCMC=config['LCMC'])

        for i, layer in enumerate(self.base.distilbert.transformer.layer):
            layer.attention.dropout = SoftDropout(p, ref=self.mask_storage['dropout_att'][i], LCMC=config['LCMC'])
            layer.ffn.dropout = SoftDropout(p, ref=self.mask_storage['dropout_ffn'][i], LCMC=config['LCMC'])

        self.base.dropout = SoftDropout(p, ref=self.mask_storage['dropout_cls'], LCMC=config['LCMC'])

        self.fc3 = nn.Linear(768, 128)
        self.fc4 = nn.Linear(128, 8)
        self.fc5 = nn.Linear(20, 20)
        self.fc6 = nn.Linear(20, 10)

        self.fc7 = nn.Linear(10, 1)
        self.LCMC = config['LCMC']

    def forward(self, x, att, mask=None):

        for key, val in self.mask_storage.items():
            if isinstance(val, dict):
                for k, v in val.items():
                    if len(v): v.pop()
            else:
                if len(val): self.mask_storage[key].pop()

        if mask is not None:
            for key, val in mask.items():
                if isinstance(val, dict):
                    for k, ref in mask[key].items():
                        if len(ref): self.mask_storage[key][k].append(ref)
                else:
                    if len(val): self.mask_storage[key].append(val)




        outputs = self.base.distilbert(
            input_ids=x,
            attention_mask=att
        )

        hidden_states = outputs.last_hidden_state  # (B, S, H)

        # CLS token
        pooled = hidden_states[:, 0]  # (B, H)

        # Original classifier path
        pooled_cls = self.base.pre_classifier(pooled)
        pooled_cls = F.relu(pooled_cls)

        pooled_cls = self.base.dropout(pooled_cls)
        logits = self.base.classifier(pooled_cls)

        # ---- Your uncertainty head ----
        unc_hidden = F.relu(self.fc3(pooled))
        unc_hidden = self.fc4(unc_hidden)

        combined = torch.cat((unc_hidden, logits), dim=-1)
        combined = F.silu(combined)

        combined = F.relu(self.fc5(combined))
        combined = F.relu(self.fc6(combined))

        unc = F.softplus(self.fc7(combined))

        return logits, unc




In [ ]:
%%writefile evaluation.py
import numpy as np
from torch import nn
import torch
from sklearn.metrics import confusion_matrix
from tqdm import tqdm

def mc_cal(model, dataloader, allowed_indices, T=32, M=10 , R=3, num_classes=10):
    model.train()  # Keep dropout active
    device = next(model.parameters()).device

    # Initialize accumulators
    class_correct = np.zeros(num_classes)
    class_counts = np.zeros(num_classes)
    class_aleatoric = np.zeros(num_classes)
    class_total = np.zeros(num_classes)

    loop = tqdm(dataloader, desc="test", leave=False)
    for data in loop:
        x = data['input_ids'].to(device)
        target = data['label'].to(device)
        att = data['attention_mask'].to(device)

        # Monte Carlo samples
        candidates = torch.randperm(M)[:R].to(device)
        probs_T = []

        for _ in range(T):
            index = candidates[torch.randint(0, R, [len(target)])]
            out = model(x, att, index)
            probs = F.softmax(out, dim=1)
            probs_T.append(probs.unsqueeze(0))  # [1, batch, num_classes]
        probs_T = torch.cat(probs_T, dim=0)  # [T, batch, num_classes]

        probs_T_s = probs_T / probs_T.sum(dim=-1, keepdim=True)
        mean_probs_s = probs_T_s.mean(dim=0)

        # Mean predictive distribution
        mean_probs = probs_T.mean(dim=0)  # [batch, num_classes]

        # --- Entropies ---
        # Aleatoric (expected entropy)

        aleatoric_all = -(probs_T_s.clamp(.00001) * probs_T_s.clamp(.00001).log2()).sum(dim=2)
        aleatoric = aleatoric_all.mean(dim=0)


        # Total (predictive) entropy
        total_ent = -(mean_probs_s.clamp(.00001) * mean_probs_s.clamp(.00001).log2()).sum(dim=1)  # [batch]

        preds = mean_probs.argmax(dim=1)

        for i in range(len(target)):
            cls = target[i].item()

            class_counts[cls] += 1
            class_aleatoric[cls] += aleatoric[i].item()
            class_total[cls] += total_ent[i].item()
            class_correct[cls] += (allowed_indices[preds[i]] == target[i]).item()

    # Compute averages
    avg_acc = class_correct / np.maximum(class_counts, 1)
    avg_aleatoric = class_aleatoric / np.maximum(class_counts, 1)
    avg_total = class_total / np.maximum(class_counts, 1)
    avg_epistemic = avg_total - avg_aleatoric

    return {
        "accuracy_per_class": avg_acc,
        "aleatoric_per_class": avg_aleatoric,
        "total_per_class": avg_total,
        "epistemic_per_class": avg_epistemic,
    }



def mc_dropout_cal_sample(model, batch, T=20, M=10, R=3):
    model.train()  # keep dropout active

    device = next(model.parameters()).device

    probs_T = []

    x = batch['input_ids'].to(device)
    att = batch['attention_mask'].to(device)

    batch_size = x.size(0)
    candidates = torch.randperm(M)[:R].to(device)
    for _ in range(T):
        index = candidates[torch.randint(0, R, [batch_size])]
        out = model(x, att, index)
        probs = F.softmax(out, dim=1)
        probs_T.append(probs.unsqueeze(0))  # [1, batch, num_classes]
    probs_T = torch.cat(probs_T, dim=0)  # [T, batch, num_classes]

    sort_cands, _ = torch.sort(candidates)
    masks = model.get_candidates(sort_cands)



    # Mean predictive distribution
    mean_probs = probs_T.mean(dim=0)  # [batch, num_classes]

    aleatoric_all = -(probs_T.clamp(.001) * probs_T.clamp(.001).log2()).sum(dim=-1)

    aleatoric = aleatoric_all.mean(dim=0)




    total_ent = -(mean_probs.clamp(.00001) * mean_probs.clamp(.00001).log2()).sum(dim=-1)  # [batch]
    epistemic = total_ent - aleatoric


    return {
        "preds": mean_probs,
        "aleatoric": aleatoric,
        "epistemic": epistemic,
        "total": total_ent,
        "total-val": total_ent,
        "masks": masks
    }



def get_uncertainty_class(pred_entropy):
    boarders = [torch.log2(torch.tensor(1.2)), torch.log2(torch.tensor(1.8)), torch.log2(torch.tensor(3)), torch.log2(torch.tensor(5))]

    uncertainty = -torch.ones((len(pred_entropy), 4), dtype=torch.float)
    for j in range(len(pred_entropy)):
        dis = torch.tensor([(pred_entropy[j] - boarders[i]) for i in range(len(boarders))])
        uncertainty[j, :] = dis > 0

    return uncertainty



@torch.no_grad()
def evaluate_student_per_class(student, teacher, test_loader, device, T=20, num_classes=10, M=10, R=3, known_unknown=0, unknown_unknown=1):
    student.eval()
    teacher.train()

    allowed_indices = []
    for i in range(num_classes):
        if i not in [known_unknown, unknown_unknown]:
            allowed_indices.append(i)


    # Initialize accumulators
    class_correct = np.zeros(num_classes)
    class_total = np.zeros(num_classes)
    class_aleatoric = np.zeros(num_classes)
    class_aleatoric_avg = np.zeros(num_classes)
    class_total_ent = np.zeros(num_classes)
    class_kl_loss = np.zeros(num_classes)

    loss_func = nn.KLDivLoss(reduction='batchmean')

    all_unc = []
    all_labels = []

    for data in test_loader:

        x = data['input_ids'].to(device)
        target = data['label'].to(device)
        att = data['attention_mask'].to(device)

        # --- Teacher MC Dropout outputs ---
        detail = mc_dropout_cal_sample(teacher, data, T=T)
        teacher_total = detail["aleatoric"]


        cnadidates, _ = torch.sort(torch.randperm(M)[:R].to(device))
        masks = teacher.get_candidates(cnadidates)

        student_output, student_unc = student(x, att, masks)
        preds = student_output.argmax(dim=1)

        student_pred = F.log_softmax(student_output, dim=1)


        for i in range(len(target)):
            cls = target[i].item()

            preds[i] = allowed_indices[preds[i]]

            class_total[cls] += 1
            class_correct[cls] += (preds[i] == target[i]).item()
            class_aleatoric[cls] += np.power(teacher_total[i].item() - student_unc[i].item(), 2)
            class_aleatoric_avg[cls] += student_unc[i].item()
            class_total_ent[cls] += teacher_total[i].item()
            class_kl_loss[cls] += loss_func(student_pred[i], detail['preds'][i])

            if target[i].item()==unknown_unknown:
                all_unc.append(student_unc[i].item())
                all_labels.append(teacher_total[i].item())

    # Compute averages
    acc_per_class = class_correct / np.maximum(class_total, 1)
    aleatoric_per_class = np.sqrt(class_aleatoric / np.maximum(class_total, 1))
    total_per_class = class_total_ent / np.maximum(class_total, 1)
    aleatoric_avg_per_class = class_aleatoric_avg / np.maximum(class_total, 1)
    class_kl_loss = class_kl_loss / np.maximum(class_total, 1)



    # Return dictionary
    return {
        "accuracy_per_class": acc_per_class,
        "aleatoric_per_class": aleatoric_per_class,
        "total_per_class": total_per_class,
        "aleatoric_avg_per_class": aleatoric_avg_per_class,
        "class_kl_loss": class_kl_loss
    }


In [ ]:
%%writefile main.py
import os

import numpy as np
import torch
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset, Dataset
import sys
from evaluation import mc_dropout_cal_sample, evaluate_student_per_class
from teacher_trainer import train_teacher
import torch.nn as nn
from collections import OrderedDict
from datasets import load_dataset
from sklearn.datasets import fetch_20newsgroups
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification
from tqdm import tqdm
from student import StudentModel
from teacher import TeacherModel




class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]
        encoding = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_len,
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].unsqueeze(-2).bool(),
            'label': torch.tensor(label, dtype=torch.long)
        }



known_unknown = 12
unknown_unknown = 13
batch_size = 32

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

config = {'model_name': "distilbert-base-uncased",
         'num_classes': 12, 'p':0.5, 'rw':.4, 'M':10, 'T':128, 'R':3,
         'device': device,
         'LCMC': True
}

num_epochs = 2


LCMC = [True, False]


for j in range(2):
    print(config)
    for _ in range(1):

        config['LCMC'] = LCMC[j]
        allowed_indiced = []
        for i in range(14):
            if i not in [unknown_unknown]:
                allowed_indiced.append(i)

        teacher = TeacherModel(config).to(device)

        for name, param in teacher.named_parameters():
            if name in teacher.state_dict().keys():
                if ("classifier" not in name) and ("dropout" not in name) and (name not in ['fc3', 'fc4', 'fc5', 'fc6', 'fc7']):
                    param.requires_grad = False

        # teacher.load_state_dict(torch.load("model/teacher_model.pth"))
        print('******** train teacher')
        train_teacher(teacher, config, known_unknown=known_unknown, unknown_unknown=unknown_unknown)

        student = StudentModel(config).to(device)


        tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')

        dataset = load_dataset("dbpedia_14")

        train_full = dataset['train']
        train_full = TextDataset(train_full['content'], train_full['label'], tokenizer, max_len=64)
        # Keep only labels 1..8 for training
        train_indices = [i for i, data in enumerate(train_full) if data['label'] in allowed_indiced]
        train_full = Subset(train_full, train_indices)

        test_ds = dataset['test']
        test_ds = TextDataset(test_ds['content'], test_ds['label'], tokenizer, max_len=64)




        student.load_state_dict(teacher.state_dict(), strict=False)



        for name, param in student.named_parameters():
            if name in teacher.state_dict().keys():
                if ("classifier" not in name) and ("dropout" not in name) and (name not in ['fc3', 'fc4', 'fc5', 'fc6', 'fc7']):
                    param.requires_grad = False


        train_loader = DataLoader(train_full, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
        test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

        optimizer = torch.optim.AdamW(
            student.parameters(),
            lr=2e-5,
            weight_decay=0.01,
            betas=(0.9, 0.999),
            eps=1e-8
        )


        scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=1, gamma=0.96)


        criterion_cls = nn.KLDivLoss(reduction='batchmean')


        gamma = .75

        print('******** train student')

        for epoch in range(num_epochs):
            student.train()
            total_loss = 0

            for data in tqdm(train_loader):
                x = data['input_ids'].to(device)
                y = data['label'].to(device)
                att = data['attention_mask'].to(device)


                with torch.no_grad():
                    detail = mc_dropout_cal_sample(teacher, data, T=config['T'])

                optimizer.zero_grad()
                output, unc = student(x, att, detail['masks'])

                output = F.log_softmax(output, dim=-1)
                loss_cls = criterion_cls(output, detail['preds'])
                loss_unc = F.mse_loss(unc.squeeze(), detail["aleatoric"])

                loss = loss_cls + gamma * loss_unc

                loss.backward()
                optimizer.step()
                total_loss += loss.item()

            scheduler.step()
            with open(f"log-{known_unknown}{unknown_unknown}-{config['p']}-{config['rw']}.txt", "a") as f:
                print(f"Epoch {epoch + 1}/{num_epochs}, Loss: {total_loss / len(train_loader):.4f}", file=f)
            print(f"Epoch {epoch + 1}/{num_epochs}, Loss: {total_loss / len(train_loader):.4f}")

            if (epoch+1) % 1 == 0:
                torch.save(student.state_dict(), "./model/student_model-1.pth")
                res = evaluate_student_per_class(student, teacher, train_loader, device, T=config['T'], num_classes=14,
                                                known_unknown=known_unknown, unknown_unknown=unknown_unknown, M=config['M'], R=config['R'])
                with open(f"log-{known_unknown}{unknown_unknown}-{config['p']}-{config['rw']}.txt", "a") as f:
                    print(res, file=f)
                print(res)

            if (epoch+1) % 1 == 0:
                res = evaluate_student_per_class(student, teacher, test_loader, device, T=config['T'], num_classes=14,
                                                known_unknown=known_unknown, unknown_unknown=unknown_unknown, M=config['M'], R=config['R'])
                with open(f"log-{known_unknown}{unknown_unknown}-{config['p']}-{config['rw']}.txt", "a") as f:
                    print(res, file=f)
                print(res)

        print("*****************---------*****************")




In [ ]:
!python main.py